# 9. GBDT Ensemble — XGB + LightGBM + CatBoost soft vote (FOC-175, phase F3)

The F3 question: **does a three-member GBDT soft-vote ensemble beat the
single-model arms on this fraud problem — and does any of it survive axes that
do not hand the test set the training customers' identities?**

Where the ladder stands (docs/FOC-174-report.md, nb7/nb8):

- **F0** — XGB baseline on the 10 transaction features: test PR-AUC 0.0532
  (chronological split).
- **F1** — base + client features (nb7 arm b): 0.2374 chronological — read
  there as customer-identity signal, not transferable demographics.
- **F2** — SCE 0.0340 / dictionary 0.1034 chronological (nb8); on the
  customer-grouped split every arm collapsed to ~0.01, next to chance.
- **F3 primary axis** — random-grouped (FOC-175): customers assigned to
  train/test by a seeded random draw — customer-disjoint, no time ordering.

The arm (`gbdt-ensemble` in the unified runner `src/fraud_pipeline.py`,
implemented in `src/arms_gbdt.py`): equal-weight **soft vote** — the mean
predicted fraud probability — of three gradient-boosted-tree classifiers on
the SAME base+client feature matrix as the xgb-client arm (column names made
LightGBM-safe; values and column order unchanged). House discipline mirrors
`fraud_pipeline.xgb_params`: fixed hyperparameters (learning_rate 0.05, depth
6, 200 trees), rare-class weighting computed from the fitting carve
(`scale_pos_weight` for XGB/LightGBM, `class_weights` for CatBoost), seed 42
everywhere, single-threaded CPU fits with `deterministic=True` (LightGBM) and
`thread_count=1` (CatBoost) — two executions of this notebook yield identical
numbers.

Protocol (nb7/nb8 discipline, driven through the runner API — never
re-implemented here): one split per axis, stratified validation carve from
TRAIN only, threshold frozen on the carve, one-shot frozen-threshold test
evaluation with percentile-bootstrap AUC intervals. Every results table below
carries test positives and the chance level (the test positive rate a random
ranking lands at) — 91 frauds total make one unreadable without the other.

In [1]:
# Runtime provenance - executed in the phase worktree venv built from the
# pinned requirements (kernel python3). Printed so the committed, executed
# notebook self-documents the exact runtime the numbers were produced on.
import platform
import sys

import catboost
import lightgbm
import numpy
import pandas
import sklearn
import xgboost

print("python:", sys.version.split()[0], "| platform:", platform.platform())
print("kernel: python3 (nbclient + WindowsSelectorEventLoopPolicy)")
for _mod in (pandas, numpy, sklearn, xgboost, lightgbm, catboost):
    print("%s: %s" % (_mod.__name__, _mod.__version__))

python: 3.11.9 | platform: Windows-10-10.0.26200-SP0
kernel: python3 (nbclient + WindowsSelectorEventLoopPolicy)
pandas: 2.3.3
numpy: 2.4.6
sklearn: 1.7.2
xgboost: 2.1.4
lightgbm: 4.7.0
catboost: 1.2.10


In [2]:
from sklearn.model_selection import StratifiedKFold, train_test_split

from arms_gbdt import make_ensemble, make_member
from fraud_pipeline import (
    ARMS,
    AXES,
    axis_split,
    cross_validate_model,
    load_enriched,
    print_comparison_table,
    register_arm,
    run_arm_on_axis,
)

# Canonical enriched frame + labels from the unified runner (the nb7/nb8 data
# section, shared by every arm — loaded once, used by everything below).
enriched, y = load_enriched()
print(
    "fraud txns: %d of %d (%.2f%%) across %d unique customers"
    % (int(y.sum()), len(y), 100 * y.mean(), enriched["customer"].nunique())
)

fraud txns: 91 of 5302 (1.72%) across 100 unique customers


C:\Users\mateu\Documents\GitHub\la-wt\Fraud-Prediction\foc-175-f3\src\funs.py:197: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  trxns_data["timestamp"] = pd.to_datetime(


## The arm through the runner — all three axes

`run_arm_on_axis` owns the whole protocol per axis: axis split → validation
carve → model (weights from the carve's y) → fit → threshold frozen on the
carve → test metrics + 1000-sample percentile-bootstrap AUC CIs. `cv=False`
here so the 5-fold refits are not paid twice — the CV evidence is produced
once, in its own cell below. The axes, in registry order:

- **random-grouped (PRIMARY)** — seeded random customer assignment,
  customer-disjoint, no time ordering;
- **grouped (stress)** — test = latest-seen customers;
- **chronological (stress)** — test strictly later than train.

In [3]:
rows = []
for axis in AXES:  # registry order: random-grouped (PRIMARY), grouped, chronological
    rows.append(run_arm_on_axis("gbdt-ensemble", axis, enriched, y, cv=False))
print_comparison_table(rows, title="gbdt-ensemble — test metrics per axis (frozen threshold)")


== gbdt-ensemble — test metrics per axis (frozen threshold) ==
          axis           arm status  test_positives  test_rows  chance_level  pr_auc  pr_auc_ci_low  pr_auc_ci_high  roc_auc  roc_auc_ci_low  roc_auc_ci_high     f1  recall_at_precision  frozen_threshold  n_features
random-grouped gbdt-ensemble     ok              13        977        0.0133  0.0145         0.0079          0.0343   0.4941          0.3560           0.6367 0.0000               0.0000            0.3256         116
       grouped gbdt-ensemble     ok              11        831        0.0132  0.0094         0.0051          0.0163   0.2879          0.1646           0.4128 0.0000               0.0000            0.8110         116
 chronological gbdt-ensemble     ok              24       1061        0.0226  0.2409         0.0884          0.4231   0.7689          0.6614           0.8690 0.2581               0.2083            0.5716         116


## CV — stratified 5-fold on the primary axis (random-grouped)

Unlike the SCE arm (whose per-fold engine refit made CV too expensive, so it
registers `supports_cv=False`), the ensemble is 3 x 200 trees on ~4k rows — a
few seconds per fold — so it ships with full CV support and the CLI runs it by
default. Class weights come from the validation carve's y (nb7 behaviour: one
weight shared by every fold clone). The folds table carries per-fold positives
and the fold chance level; read CV as stability evidence, never as the
headline — the folds mix customers and periods, the test axes do not.

In [4]:
X_raw = ARMS["gbdt-ensemble"]["build_features"](enriched)
train_idx, _ = axis_split("random-grouped", enriched, y)
X_tr, y_tr = X_raw.loc[train_idx], y.loc[train_idx]

# Same carve run_arm_on_split uses: the class weights read the fitting carve only.
_, _, y_fit, _ = train_test_split(X_tr, y_tr, test_size=0.25, random_state=42, stratify=y_tr)
cv = cross_validate_model(make_ensemble(y_fit), X_tr, y_tr, k=5, stratified=True, random_state=42)

# Fold context (positives + chance on every results table): same splitter,
# same seed -> exactly the folds cross_validate_model drew.
fold_pos, fold_rows = [], []
for _, te_i in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X_tr, y_tr):
    fold_pos.append(int(y_tr.iloc[te_i].sum()))
    fold_rows.append(len(te_i))
folds_table = cv["folds"].assign(
    fold_positives=fold_pos,
    fold_rows=fold_rows,
    fold_chance=[p / n for p, n in zip(fold_pos, fold_rows)],
)
print(folds_table.round(4).to_string(index=False))
summary = cv["summary"]
print(
    "\nCV PR-AUC %.4f+-%.4f | CV ROC-AUC %.4f+-%.4f | train: %d positives in %d rows"
    % (
        summary["pr_auc_mean"], summary["pr_auc_std"],
        summary["roc_auc_mean"], summary["roc_auc_std"],
        int(y_tr.sum()), len(y_tr),
    )
)

 fold  precision  recall     f1  accuracy  pr_auc  roc_auc  fold_positives  fold_rows  fold_chance
    1     0.6875  0.7333 0.7097    0.9896  0.6520   0.8296              15        865       0.0173
    2     0.6000  0.6000 0.6000    0.9861  0.6749   0.9218              15        865       0.0173
    3     0.9167  0.6875 0.7857    0.9931  0.8743   0.9961              16        865       0.0185
    4     0.8333  0.6250 0.7143    0.9908  0.5984   0.8345              16        865       0.0185
    5     0.6250  0.6250 0.6250    0.9861  0.7663   0.9892              16        865       0.0185

CV PR-AUC 0.7132+-0.1086 | CV ROC-AUC 0.9142+-0.0805 | train: 78 positives in 4325 rows


## Per-member vs ensemble — primary axis only

The ensembling question at this scale: does averaging three GBDTs beat its
own members? The three members run under the exact runner protocol as
one-off diagnostic arms registered in THIS kernel only (never part of the
committed registry or the results file — `register_arm` is the runner's
extension point, so no parallel protocol can drift): same feature matrix,
same carve, same frozen-threshold procedure. The ensemble row is the
random-grouped row from the table above — same split, same protocol.

In [5]:
build_features = ARMS["gbdt-ensemble"]["build_features"]


def _member_factory(member_name):
    # Closure factory so each diagnostic arm pins its own member name.
    return lambda y_fit: make_member(member_name, y_fit)


for member_name in ("xgb", "lgbm", "catboost"):
    register_arm(
        "gbdt-%s" % member_name,
        "diagnostic (nb9 kernel only): lone %s member of the gbdt-ensemble arm" % member_name,
        make_model=_member_factory(member_name),
        build_features=build_features,
    )

member_rows = [
    run_arm_on_axis("gbdt-xgb", "random-grouped", enriched, y, cv=False),
    run_arm_on_axis("gbdt-lgbm", "random-grouped", enriched, y, cv=False),
    run_arm_on_axis("gbdt-catboost", "random-grouped", enriched, y, cv=False),
    next(r for r in rows if r["axis"] == "random-grouped"),  # ensemble: same axis, same protocol
]
print_comparison_table(
    member_rows, title="per-member vs soft vote — random-grouped (PRIMARY axis)"
)

ensemble_row, best_member = member_rows[-1], max(member_rows[:3], key=lambda r: r["pr_auc"])
delta = ensemble_row["pr_auc"] - best_member["pr_auc"]
EPSILON = 0.001
verdict = (
    "ensembling HELPS" if delta > EPSILON
    else "ensembling HURTS" if delta < -EPSILON
    else "NO SIGNAL — soft voting does not move the best member"
)
print(
    "ensemble vs best lone member (test PR-AUC): %+.4f -> %s (delta is noise-dominated at this test size)"
    % (delta, verdict)
)


== per-member vs soft vote — random-grouped (PRIMARY axis) ==
          axis           arm status  test_positives  test_rows  chance_level  pr_auc  pr_auc_ci_low  pr_auc_ci_high  roc_auc  roc_auc_ci_low  roc_auc_ci_high  f1  recall_at_precision  frozen_threshold  n_features
random-grouped gbdt-ensemble     ok              13        977        0.0133  0.0145         0.0079          0.0343   0.4941          0.3560           0.6367 0.0                  0.0            0.3256         116
random-grouped      gbdt-xgb     ok              13        977        0.0133  0.0144         0.0079          0.0300   0.4844          0.3334           0.6425 0.0                  0.0            0.8776         116
random-grouped     gbdt-lgbm     ok              13        977        0.0133  0.0213         0.0104          0.0574   0.6022          0.4559           0.7445 0.0                  0.0            0.3548         116
random-grouped gbdt-catboost     ok              13        977        0.0133  0.0116 

### Interpretation (read after the tables — null results are findings)

- **Noise budget first.** 91 frauds total; the three splits put 13
  (random-grouped), 11 (grouped) and 24 (chronological) test positives in
  play — printed on every table row next to its chance level. At that size a
  single swapped fraud moves test PR-AUC by hundredths and the bootstrap CIs
  span a wide band around every point estimate: deltas under ~0.05 PR-AUC
  between arms are noise, not signal.
- **Read every number against its chance level.** Chance PR-AUC equals the
  test positive rate: 0.0133 on random-grouped, 0.0132 on grouped, 0.0226 on
  chronological. The ensemble's test PR-AUC (first table) and the per-member
  numbers (third table) must be judged by their distance from those baselines
  AND by their printed CIs — an interval that still covers the chance level
  means the model is indistinguishable from a random ranking on that split,
  whatever the point estimate suggests.
- **Does ensembling help at this scale?** The verdict line under the
  per-member table compares the soft vote against its best lone member under
  the identical protocol. With correlated members trained on ~4k rows and
  ~78 training positives, soft voting mostly averages near-identical
  rankings — expect a small delta either way, and treat any delta inside the
  bootstrap band as a NULL RESULT: at this scale the honest answer is "no
  evidence ensembling helps", which is a finding, not a failure.
- **Axis structure.** random-grouped is PRIMARY (customer-disjoint but
  temporally unbiased); grouped and chronological are stress tests that also
  remove recency overlap. nb7/nb8 showed the client-feature lift is
  customer-identity signal that collapses once test customers are disjoint
  from train; the ensemble consumes the same feature set, so it inherits that
  collapse — compare its PR-AUC per axis against the chance level above.
- **CV vs test gap.** The 5 CV folds are stratified random within TRAIN —
  they mix customers and periods — while every test axis is customer-disjoint
  (and the two stress axes also time-shifted). Read the CV table as stability
  evidence only; a large CV-vs-test gap is expected, not a bug.
- **Determinism.** Seeds 42 everywhere (splits, carve, members, bootstrap);
  single-threaded CPU fits with LightGBM `deterministic=True` and CatBoost
  `thread_count=1`. Re-running this notebook reproduces every number exactly.

## Summary

- `src/arms_gbdt.py` implements the F3 `gbdt-ensemble` arm: equal-weight soft
  vote of XGB + LightGBM + CatBoost on the base+client feature matrix, with
  per-member rare-class weights from the fitting carve, fixed seeds and
  CPU-deterministic fits.
- The arm is registered in the unified runner (`src/fraud_pipeline.py`) with
  full CV support and is exercised end-to-end here through `run_arm_on_axis`
  on all three axes; the three members ran under the identical protocol on
  the primary axis as kernel-only diagnostic arms.
- The verdicts are read from the tables above against their chance levels
  and bootstrap CIs; the committed CLI run of the arm appends the same
  protocol's numbers (with the CV summary columns) to
  `results/fraud_pipeline_results.jsonl`.